## Setup and Import

In [ ]:
import importlib
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../"))

# Force complete module reload by removing from cache
if "utils.gref_pipeline.georef" in sys.modules:
    del sys.modules["utils.gref_pipeline.georef"]
if "utils.gref_pipeline" in sys.modules:
    del sys.modules["utils.gref_pipeline"]
if "gref_pipeline.georef" in sys.modules:
    del sys.modules["gref_pipeline.georef"]

from utils.gref_pipeline import georef
from gref_pipeline import config

importlib.reload(georef)
from utils.gref_pipeline.georef import *

print("✅ Module reloaded with cache cleared!")

## Load Data

In [ ]:
# Load 028 transect
transect = load_transect(r"E:\mjosa_new_oct_2025\use_gref4hsi\028\output")
transect.list_files()

# Select all 5 files from transect 028
cube = transect.select_files(
    [
        "rad_uhi_20241029_125028_1",
        "rad_uhi_20241029_125028_2",
        "rad_uhi_20241029_125028_3",
        "rad_uhi_20241029_125028_4",
        "rad_uhi_20241029_125028_5",
    ]
)
cube.describe()

In [ ]:
# Apply illumination correction
cube.apply_illumination_correction_v2()

## Import and Map ROIs

Load ROIs from dev27 and map them to main classes.

In [ ]:
# Import ROIs
cube.import_rois("./ROIs/028_new.json")
cube.list_rois()

In [ ]:
# Define ROI mapping to main classes
roi_mapping = {
    # Binary classification ROIs
    "sediment": "sediment",
    "rust": "rust",
    # Multi-class: dark bombs (numbered dark areas around bombs)
    "1_dark": "dark_bomb",
    "2_dark": "dark_bomb",
    "3_dark": "dark_bomb",
    # Multi-class: dark pits (separate dark pit class)
    "dark_pits": "dark_pit",
    "dark_pits_multiple": "dark_pit",  # Additional dark pit ROI (same class as dark_pits)
    # Multi-class: halos
    "1_halo": "halo",
    "2_halo": "halo",
    "3_halo": "halo",
}

# Define training ROIs for each classification mode
training_rois_binary = ["sediment", "rust"]

training_rois_multiclass = [
    "sediment",
    "rust",
    "1_dark",
    "2_dark",
    "3_dark",
    "dark_pits",
    "dark_pits_multiple",  # Additional dark pit ROI (same class as dark_pits)
    "1_halo",
    "2_halo",
    "3_halo",
]

print("📋 ROI Mapping:")
for roi, mapped in roi_mapping.items():
    if roi in cube.roi_collection:
        print(f"   {roi} → {mapped} ({len(cube.roi_collection[roi])} pixels)")

print(f"\n🎯 Binary classification ROIs: {training_rois_binary}")
print(f"🎯 Multi-class classification ROIs: {training_rois_multiclass}")

## Visualize ROIs on Full Transect

In [ ]:
%matplotlib inline

# Plot all training ROIs for multi-class
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    figsize=(5, 50),
    show_file_boundaries=False,
    roi_collection=training_rois_multiclass,
    roi_legend_loc="outside",
    roi_marker_size=2,
    roi_legend_markersize=50,
    roi_marker_edgewidth=0,
)

## Define Crop Regions

Define the 4-5 cropped regions where classification will be applied.

In [ ]:
# Define crop regions (UPDATED coordinates from user)
crop_regions = [
    {
        "name": "Crop 1 (Bomb 2 - Rust)",
        "track": 1258,
        "slit": 250,  # UPDATED from 212
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 2 (Bomb 3)",
        "track": 5592,
        "slit": 700,  # UPDATED from 765
        "width": 500,
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 3 (Bomb 1)",
        "track": 5160,
        "slit": 613,
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
    {
        "name": "Crop 4 (Dark pits)",
        "track": 717,
        "slit": 385,
        "width": 500,  # UPDATED from 400
        "aspect_ratio": 4,  # UPDATED from 3.5
    },
]

print(f"📍 Defined {len(crop_regions)} crop regions (UPDATED):")
for i, crop in enumerate(crop_regions, 1):
    print(
        f"   {i}. {crop['name']}: track={crop['track']}, slit={crop['slit']}, width={crop['width']}"
    )

## Visualize Crop Regions with ROIs

In [ ]:
# TEST: Plot crop that extends beyond image boundary
# Should show GLOBAL coordinates in extent (can be negative)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    figsize=(30, 8.32),  # FIXED - was (8, 8)
    crop_center_track=1258,
    crop_center_slit=150,  # Near edge - crop will extend beyond boundary
    crop_width=600,
    crop_aspect_ratio=4,  # FIXED - was 3.5
    display_aspect_ratio=4.0,  # ADDED
    show_file_boundaries=False,
)
print(
    "\n✅ Expected: Slit extent should be [-150, 450] (centered at 150 with width 600)"
)
print("   Even though image only has slits [0, 968], extent shows requested range")

In [ ]:
# Visualize each crop region with relevant ROIs (UPDATED settings)
for crop in crop_regions:
    print(f"\n📷 {crop['name']}")
    cube.plot_rgb(
        use_corrected=True,
        flip_axes=True,
        flip_horizontal=True,
        figsize=(30, 8.32),  # UPDATED
        crop_center_track=crop["track"],
        crop_center_slit=crop["slit"],
        crop_width=crop["width"],
        crop_aspect_ratio=crop["aspect_ratio"],
        display_aspect_ratio=4.0,  # UPDATED - Controls display stretching
        show_file_boundaries=False,
        roi_collection=training_rois_multiclass,
        roi_legend_loc="outside",
        roi_marker_size=2,  # FIXED - was 200
        roi_legend_markersize=50,
        roi_marker_edgewidth=0,
    )

### Step 1: Crop Wavelengths (490-680 nm)

In [ ]:
# Apply smoothing BEFORE wavelength filter (force recompute to avoid cache issues)
cube.apply_spectral_smoothing(
    method="gaussian",
    gaussian_sigma=5.0,
)

In [ ]:
# Crop wavelengths to 490-700 nm (UPDATED)
cube.apply_wavelength_filter(wavelength_range=(490, 700))

print(f"✅ Wavelength cropping complete")
print(f"   Cropped shape: {cube.data_corrected.shape}")
print(f"   Wavelength range: {cube.wavelengths[0]:.1f} - {cube.wavelengths[-1]:.1f} nm")

### Step 3: L2 Normalization (COMMENTED OUT FOR NOW)

In [ ]:
# # Apply L2 normalization (unit-length spectra)
# cube.apply_spectral_normalization(method="l2")

# print(f"✅ L2 normalization complete")
# print(
#     f"   Value range: [{cube.data_corrected.min():.4f}, {cube.data_corrected.max():.4f}]"
# )

### Visualize Preprocessed Spectra

In [ ]:
# Plot preprocessed spectra for multi-class training ROIs
cube.plot_spectrum(
    roi_names=training_rois_multiclass,
    use_corrected=True,
    wavelength_range=(
        490,
        700,
    ),  # Filter to 490-680 nm (don't rely on apply_wavelength_filter)
    # wavelength_smoothing=1,  # No additional smoothing
    normalize_method=None,  # No additional normalization
    legend_loc="outside",
    use_inline_labels=False,
    show_std=True,
)

## 🐛 **DATA CORRUPTION ISSUE**

The `cube` data is now corrupted because:
1. You ran `apply_wavelength_filter()` which modified `self.wavelengths` to 108 elements
2. You reloaded the module, which reset the methods but NOT the data
3. Now `self.wavelengths` has 108 elements but `self.data_corrected` has 210 wavelengths

**FIX:** Run cells 4-6 again to reload the transect with fresh data (starting from "Load 028 transect")

In [ ]:
# DEBUG: Check data shapes after wavelength filtering
print("🔍 DEBUG: Data shapes after apply_wavelength_filter()")
print(f"   self.wavelengths: {len(cube.wavelengths)} elements")
print(f"   self.data shape: {cube.data.shape}")
print(f"   self.data_corrected shape: {cube.data_corrected.shape}")
print(f"\n   Expected: All should have 108 wavelengths (490-680 nm)")
print(f"   Issue: data_corrected might have 108 but data might still have 210!")

## Classify Crop Regions (Binary)

Apply classification only to the cropped regions of interest.

In [ ]:
# Helper function to classify a single crop region
def classify_crop_region(cube, crop, confidence_threshold=0.5):
    """Classify a cropped region and return results."""
    # Calculate crop bounds
    half_width_slit = crop["width"] // 2
    half_width_track = int(crop["width"] / crop["aspect_ratio"] / 2)

    crop_track_min = max(0, crop["track"] - half_width_track)
    crop_track_max = min(cube.data_corrected.shape[0], crop["track"] + half_width_track)
    crop_slit_min = max(0, crop["slit"] - half_width_slit)
    crop_slit_max = min(cube.data_corrected.shape[1], crop["slit"] + half_width_slit)

    print(f"\n🔍 Classifying: {crop['name']}")
    print(
        f"   Crop bounds: track [{crop_track_min}:{crop_track_max}], slit [{crop_slit_min}:{crop_slit_max}]"
    )

    # Classify the segment
    classification_results = cube.classify_segment(
        segment_start=crop_track_min,
        segment_end=crop_track_max - 1,
        use_corrected=True,
        confidence_threshold=confidence_threshold,
        quiet=False,
    )

    # Print pixel counts
    print(f"\n📊 Pixel counts for {crop['name']}:")
    for class_name in classification_results["class_names"]:
        count = np.sum(classification_results["classification_map"] == class_name)
        percentage = 100.0 * count / classification_results["classification_map"].size
        print(f"   {class_name}: {count} pixels ({percentage:.1f}%)")

    return classification_results, crop_track_min, crop_track_max

---

# PART 2: Multi-Class Classification (Sediment vs Rust vs Dark vs Halo)

---

## Create Mapped Training ROIs

Map ROIs to main classes for multi-class training.

In [ ]:
# Create new ROI collection with mapped class names
def create_mapped_roi_collection(cube, roi_list, roi_mapping):
    """Create ROI collection with mapped class names."""
    mapped_rois = {}

    for roi_name in roi_list:
        if roi_name in cube.roi_collection:
            mapped_name = roi_mapping.get(roi_name, roi_name)

            # Add pixels to mapped class (merge if class already exists)
            if mapped_name not in mapped_rois:
                mapped_rois[mapped_name] = []
            mapped_rois[mapped_name].extend(cube.roi_collection[roi_name])

    # Remove duplicates
    for class_name in mapped_rois:
        mapped_rois[class_name] = list(set(mapped_rois[class_name]))

    return mapped_rois


# Create mapped ROI collection
mapped_training_rois = create_mapped_roi_collection(
    cube, training_rois_multiclass, roi_mapping
)

print("📋 Mapped training ROIs for multi-class:")
for class_name, pixels in mapped_training_rois.items():
    print(f"   {class_name}: {len(pixels)} pixels")

# Store mapped ROIs temporarily for training
cube._original_roi_collection = cube.roi_collection.copy()  # Backup original ROIs
cube.roi_collection = mapped_training_rois  # Replace with mapped ROIs

## Train SVM (Multi-Class: Sediment vs Rust vs Dark vs Halo)

In [ ]:
# Train SVM with spatial cross-validation - MULTI-CLASS MODE
# BUG FIX: Use None for segment_start/end to include ALL data (not exclude)
cv_results_multiclass = cube.train_svm_with_cv(
    training_rois=list(mapped_training_rois.keys()),  # Use mapped class names
    segment_start=None,  # None = use all data
    segment_end=None,  # None = use all data
    wavelength_range=None,  # Already preprocessed
    cv_folds=5,
    use_corrected=True,
    svm_kernel="rbf",
    optimize_params=True,
    add_brightness_feature=False,
    use_intensity_only=False,
    quiet=False,
    closing_radius=50,  # INCREASED from 5 to 50 - forces connections between scattered pixels
)

# Print summary
print(f"\n" + "=" * 60)
print(f"📊 MULTI-CLASS CLASSIFICATION CROSS-VALIDATION SUMMARY")
print(f"=" * 60)
print(
    f"Accuracy:  {cv_results_multiclass['cv_mean_metrics']['accuracy_mean']:.3f} ± {cv_results_multiclass['cv_mean_metrics']['accuracy_std']:.3f}"
)
print(
    f"Precision: {cv_results_multiclass['cv_mean_metrics']['precision_mean']:.3f} ± {cv_results_multiclass['cv_mean_metrics']['precision_std']:.3f}"
)
print(
    f"Recall:    {cv_results_multiclass['cv_mean_metrics']['recall_mean']:.3f} ± {cv_results_multiclass['cv_mean_metrics']['recall_std']:.3f}"
)
print(
    f"F1 Score:  {cv_results_multiclass['cv_mean_metrics']['f1_mean']:.3f} ± {cv_results_multiclass['cv_mean_metrics']['f1_std']:.3f}"
)

print(f"\n🎯 Best hyperparameters:")
print(f"  C = {cv_results_multiclass['best_params']['C']}")
print(f"  gamma = {cv_results_multiclass['best_params']['gamma']}")

print(f"\n📍 Training pixels used:")
for class_name, count in cv_results_multiclass["training_pixels_per_class"].items():
    print(f"  {class_name}: {count} pixels")

In [ ]:
# Plot spatial groups for multi-class
if (
    "spatial_groups_roi_collection" in cv_results_multiclass
    and cv_results_multiclass["spatial_groups_roi_collection"]
):
    print("\n📊 Spatial Groups Created for Multi-Class Classification:")
    for group_name, pixels in sorted(
        cv_results_multiclass["spatial_groups_roi_collection"].items()
    ):
        print(f"   {group_name}: {len(pixels)} pixels")

    cube.plot_georef(
        use_corrected=True,
        figsize=(40, 10),
        roi_collection=cv_results_multiclass["spatial_groups_roi_collection"],
        roi_marker_size=3,
        roi_legend_loc="outside",
        roi_marker_edgewidth=0,
        roi_legend_markersize=40,
    )
else:
    print("⚠️ No spatial groups found in cv_results_multiclass.")

In [ ]:
# Restore original ROI collection
cube.roi_collection = cube._original_roi_collection
del cube._original_roi_collection

print("✅ Original ROI collection restored")

## Classify Crop Regions (Multi-Class)

In [ ]:
# Classify all crop regions (multi-class mode)
multiclass_crop_results = []

for crop in crop_regions:
    results, track_min, track_max = classify_crop_region(
        cube, crop, confidence_threshold=0.5
    )
    multiclass_crop_results.append(
        {
            "crop": crop,
            "results": results,
            "track_min": track_min,
            "track_max": track_max,
        }
    )

In [ ]:
# Convert multi-class classification results to ROI format and plot with plot_rgb
print(
    "📊 Plotting multi-class classification results with plot_rgb (includes uncertain class)..."
)

# For each crop, convert classification_map to ROI collection
for crop_result in multiclass_crop_results:
    crop = crop_result["crop"]
    results = crop_result["results"]
    track_min = crop_result["track_min"]

    # Create ROI collection from classification map
    classification_rois = {}
    for class_name in results["class_names"]:
        mask = results["classification_map"] == class_name
        rows, cols = np.where(mask)
        # Convert to global coordinates (add track_min offset)
        pixels = [(col, row + track_min) for row, col in zip(rows, cols)]
        if len(pixels) > 0:
            classification_rois[class_name] = pixels

    print(f"\n📍 {crop['name']}")
    print(f"   ROIs created: {list(classification_rois.keys())}")
    print(f"   Pixel counts:")
    for class_name in sorted(classification_rois.keys()):
        print(f"      {class_name}: {len(classification_rois[class_name])} pixels")

    # Plot with plot_rgb showing the classified region
    cube.plot_rgb(
        use_corrected=True,
        flip_axes=True,
        flip_horizontal=True,
        figsize=(30, 8.32),
        crop_center_track=crop["track"],
        crop_center_slit=crop["slit"],
        crop_width=crop["width"],
        crop_aspect_ratio=crop["aspect_ratio"],
        display_aspect_ratio=4.0,
        show_file_boundaries=False,
        roi_collection=classification_rois,
        roi_legend_loc="outside",
        roi_marker_size=2,  # FIXED - was 1, now 2 for solid pixels (s=2^2*50=200)
        roi_legend_markersize=50,  # FIXED - was 1
        roi_marker_edgewidth=0,
    )

### 🎨 Proper Classification Visualization

**Previous approach was WRONG:** Using ROI markers creates artificial symbols stacked on top of RGB image.

**Correct approach:** Create a true classified image where each pixel has the color of its class - like any normal image!

In [ ]:
def plot_classification_as_image(
    cube,
    crop,
    classification_map,
    class_names,
    figsize=(30, 8.32),
    flip_axes=True,
    flip_horizontal=True,
):
    """
    Plot classification results as a proper classified image (not ROI markers).
    Each pixel gets the color of its class - like a normal image.
    """
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    import numpy as np

    # Define colors for each class
    class_colors = {
        "sediment": [0.6, 0.4, 0.2],  # Brown
        "rust": [0.8, 0.2, 0.1],  # Red-orange
        "dark_bomb": [0.1, 0.1, 0.1],  # Dark gray/black
        "dark_pit": [0.2, 0.2, 0.2],  # Gray
        "halo": [0.9, 0.9, 0.5],  # Yellow
        "uncertain": [0.5, 0.5, 0.5],  # Medium gray
    }

    # Create RGB image from classification map
    height, width = classification_map.shape
    rgb_image = np.zeros((height, width, 3))

    for class_name in class_names:
        mask = classification_map == class_name
        color = class_colors.get(class_name, [0.5, 0.5, 0.5])  # Default gray
        rgb_image[mask] = color

    # Apply flipping to match plot_rgb behavior
    # Step 1: Transpose to match plot_rgb's initial .T.copy()
    rgb_image = rgb_image.transpose(1, 0, 2)  # (slits, tracks, 3)

    # Step 2: Apply flip_axes (like plot_rgb does)
    if flip_axes:
        rgb_image = rgb_image.transpose(1, 0, 2)  # Back to (tracks, slits, 3)

    # Step 3: Apply flip_horizontal (like plot_rgb does)
    if flip_horizontal:
        rgb_image = np.flip(rgb_image, axis=1)  # Flip along axis 1

    # Calculate extent in global coordinates
    half_width_slit = crop["width"] // 2
    half_width_track = int(crop["width"] / crop["aspect_ratio"] / 2)

    slit_min = crop["slit"] - half_width_slit
    slit_max = crop["slit"] + half_width_slit
    track_min = crop["track"] - half_width_track
    track_max = crop["track"] + half_width_track

    if flip_axes:
        # When axes flipped: x=slit, y=track (matches plot_rgb)
        extent = [slit_min, slit_max, track_min, track_max]
        xlabel, ylabel = "Slit Pixel Index", "Track Index"
    else:
        # Normal: x=track, y=slit
        extent = [track_min, track_max, slit_min, slit_max]
        xlabel, ylabel = "Track Index", "Slit Pixel Index"

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    # Display the classified image
    im = ax.imshow(
        rgb_image,
        extent=extent,
        origin="lower",
        aspect=crop.get("display_aspect_ratio", 4.0),
        interpolation="nearest",
    )

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f"Classification: {crop['name']}", fontsize=14, fontweight="bold")

    # Create legend with class colors
    legend_patches = []
    for class_name in class_names:
        count = np.sum(classification_map == class_name)
        color = class_colors.get(class_name, [0.5, 0.5, 0.5])
        label = f"{class_name} ({count} px)"
        legend_patches.append(mpatches.Patch(color=color, label=label))

    ax.legend(
        handles=legend_patches,
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        frameon=True,
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    return fig, ax


# Plot multi-class classification results as proper images
print("📊 Plotting multi-class classification as IMAGES (not ROI markers)...")

for crop_result in multiclass_crop_results:
    crop = crop_result["crop"]
    results = crop_result["results"]

    print(f"\n📍 {crop['name']}")
    print(f"   Classes: {results['class_names']}")

    # CRITICAL FIX: Crop the classification_map to the slit range!
    # classify_segment only crops track dimension, not slit dimension
    half_width_slit = crop["width"] // 2
    crop_slit_min = max(0, crop["slit"] - half_width_slit)
    crop_slit_max = min(
        results["classification_map"].shape[1], crop["slit"] + half_width_slit
    )

    # Crop the classification map to the slit range
    cropped_classification_map = results["classification_map"][
        :, crop_slit_min:crop_slit_max
    ]

    print(f"   Original shape: {results['classification_map'].shape}")
    print(
        f"   Cropped to slit [{crop_slit_min}:{crop_slit_max}]: {cropped_classification_map.shape}"
    )
    print(f"   Pixel counts:")
    for class_name in results["class_names"]:
        count = np.sum(cropped_classification_map == class_name)
        percentage = 100.0 * count / cropped_classification_map.size
        print(f"      {class_name}: {count} pixels ({percentage:.1f}%)")

    # Plot as proper classified image with CROPPED data
    plot_classification_as_image(
        cube,
        crop,
        cropped_classification_map,  # Use CROPPED classification map!
        results["class_names"],
        figsize=(30, 7.18),
        flip_axes=True,
        flip_horizontal=True,
    )

In [ ]:
# Test: Compare plot_rgb vs plot_classification_as_image
print("Testing zoom/extent comparison...")

# Use the first crop region for testing
test_crop = crop_regions[0]
test_results = multiclass_crop_results[0]["results"]

print(f"\nTest crop parameters:")
print(f"  Track: {test_crop['track']}, Slit: {test_crop['slit']}")
print(f"  Width: {test_crop['width']}, Aspect ratio: {test_crop['aspect_ratio']}")

# CRITICAL: Crop the classification_map to match the slit range!
# classify_segment only crops track dimension, not slit dimension
half_width_slit = test_crop["width"] // 2
crop_slit_min = max(0, test_crop["slit"] - half_width_slit)
crop_slit_max = min(
    test_results["classification_map"].shape[1], test_crop["slit"] + half_width_slit
)

# Crop the classification map to the slit range
cropped_classification_map = test_results["classification_map"][
    :, crop_slit_min:crop_slit_max
]

print(f"\n🔍 DEBUG: Classification map shapes:")
print(f"   Original: {test_results['classification_map'].shape}")
print(
    f"   Cropped to slit [{crop_slit_min}:{crop_slit_max}]: {cropped_classification_map.shape}"
)

# 1. Reference: plot_rgb with same parameters
print("\n1. Plotting with plot_rgb (reference)...")
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=test_crop["track"],
    crop_center_slit=test_crop["slit"],
    crop_width=test_crop["width"],
    crop_aspect_ratio=test_crop["aspect_ratio"],
    display_aspect_ratio=4.0,
    figsize=(30, 8.32),
)

# 2. Custom: plot_classification_as_image with same parameters
print("\n2. Plotting with plot_classification_as_image (custom)...")
plot_classification_as_image(
    cube,
    test_crop,
    cropped_classification_map,  # Use CROPPED classification map!
    test_results["class_names"],
    figsize=(30, 8.32),
    flip_axes=True,
    flip_horizontal=True,
)

print("\n✓ Both plots generated. Check if they have the same zoom level and extent!")

---

## 💾 SAVE CLASSIFICATION RESULTS (Run this once!)

Save the classification results so you can test plotting in dev31 without rerunning the pipeline.

In [ ]:
import pickle
import os

# Create saved_data directory if it doesn't exist
os.makedirs("./saved_data", exist_ok=True)

# Save classification results
save_data = {
    "multiclass_crop_results": multiclass_crop_results,
    "crop_regions": crop_regions,
}

save_file = "./saved_data/classification_results_028.pkl"
with open(save_file, "wb") as f:
    pickle.dump(save_data, f)

print(f"✅ Classification results saved to {save_file}")
print(f"   {len(multiclass_crop_results)} crop results")
print(f"\n📝 Now you can use dev31 for quick plotting tests!")

## 💾 SAVE SVM MODEL (for dev32)

Save the trained SVM model so it can be applied to other transects (e.g., 057 in dev32).

In [ ]:
# Debug: Check available keys in cv_results_multiclass
print("🔍 Debug: Available keys in cv_results_multiclass:")
print(cv_results_multiclass.keys())
print(
    "\nThis will help us see if it's 'trained_model', 'best_model', or something else."
)

In [ ]:
import pickle
import os

# Create saved_data directory if it doesn't exist
os.makedirs("./saved_data", exist_ok=True)

# Prepare model data for saving
model_save_data = {
    "model": cv_results_multiclass["model"],  # FIXED: correct key is "model"
    "label_encoder": cv_results_multiclass[
        "label_encoder"
    ],  # ADDED: needed for classification
    "class_names": list(mapped_training_rois.keys()),  # Class names
    "best_params": cv_results_multiclass["best_params"],  # C and gamma
    "cv_metrics": cv_results_multiclass["cv_mean_metrics"],  # Performance metrics
    "preprocessing": {
        "smoothing_method": "gaussian",
        "smoothing_sigma": 5.0,
        "wavelength_range": (490, 700),  # nm
        "normalization": None,  # No L2 normalization
    },
    "training_info": {
        "transect": "028",
        "num_training_pixels": cv_results_multiclass["training_pixels_per_class"],
        "roi_mapping": roi_mapping,
    },
}

# Save the model
model_file = "./saved_data/svm_model_028_multiclass.pkl"
with open(model_file, "wb") as f:
    pickle.dump(model_save_data, f)

print(f"✅ SVM model saved to {model_file}")
print(f"\n📊 Model Info:")
print(f"   Classes: {model_save_data['class_names']}")
print(f"   Best C: {model_save_data['best_params']['C']}")
print(f"   Best gamma: {model_save_data['best_params']['gamma']}")
print(f"   Accuracy: {model_save_data['cv_metrics']['accuracy_mean']:.3f}")
print(f"\n📝 Now you can use this model in dev32 to classify transect 057!")